In [22]:
import numpy as np
import pandas as pd
import openmatrix as omx
from pathlib import Path
from dbfread import DBF
import copy
from datetime import datetime
import os
import geopandas as gpd
from libpysal.weights import KNN


In [23]:
# import global TDM functions
import sys

sys.path.insert(0, "../Resources/2-Python/global-functions")
import BigQuery

client = BigQuery.getBigQueryClient_Confidential2023UtahHTS()


## Inputs


In [24]:
path_inputs = Path.cwd() / "inputs"
path_outputs = Path.cwd() / "outputs"
path_start_files = path_inputs / "input_files/C28"

path_coeff_file = path_inputs / "coefficients_cal8.block"
path_se_file = path_start_files / "SE_File.dbf"
path_pa_file = path_start_files / "pa_initial.dbf"
path_tel_file = path_start_files / "telecommute.dbf"
path_hh_files = [
    path_start_files / f"HH{i}_PercTrips_segment_hbw.dbf" for i in range(1, 7)
]
path_taz_file = path_start_files / "WFv1000_TAZ.dbf"
path_taz_shpfile = path_start_files / "WFv1000_TAZ.shp"
path_dens_file = path_start_files / "IntersectionDensity.dbf"

# Skims and Logsums (assuming they are already converted to OMX in this folder)
path_skm_hwy = path_start_files / "skm_auto_Pk.omx"
path_skm_walk = path_start_files / "Best_Walk_Skims.omx"
path_logsums = path_start_files / "HBW_logsums_Pk.omx"
path_skm_transit_pk = path_start_files / "1Skm_TotTransitTime_Pk.omx"
path_skm_transit_ok = path_start_files / "1Skm_TotTransitTime_Ok.omx"

# We load this once here just to get the segment names for loading matrices
# initial_coeffs = load_block_coefficients(path_coeff_file)
# segments = list(initial_coeffs.keys())


In [25]:
used_zones = 3629
dummy_zones_str = "3563-3600"
external_zones_str = "3601-3629"

# read dbf files
df_se_raw = pd.DataFrame(iter(DBF(path_se_file)))
df_pa = pd.DataFrame(iter(DBF(path_pa_file)))
df_tel = pd.DataFrame(iter(DBF(path_tel_file)))
df_hh = [pd.DataFrame(iter(DBF(f))) for f in path_hh_files]
df_taz = pd.DataFrame(iter(DBF(path_taz_file)))
gdf_taz = gpd.read_file(path_taz_shpfile)
df_dens = pd.DataFrame(iter(DBF(path_dens_file)))

# Create dummy/external mask
mask_external_dummys = np.zeros(used_zones, dtype=bool)
# apply_range_mask(mask_external_dummys, dummy_zones_str)
# apply_range_mask(mask_external_dummys, external_zones_str)


In [7]:
hts_hh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.hh"
).to_dataframe()
hts_person_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.person"
).to_dataframe()
hts_trip_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.trip_linked"
).to_dataframe()
hts_veh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.vehicle"
).to_dataframe()


c:\Users\cday\Anaconda3\envs\base0426\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


# Estimation Process 1


## Observed Data


In [ ]:
import numpy as np
import pandas as pd

hts_trip_23_merge = hts_trip_23.copy()
hts_trip_23_merge = hts_trip_23_merge[
    [
        "unique_id",
        "hh_id",
        "person_id",
        "vehicle_id",
        "pCO_TAZID_USTMv4",
        "aCO_TAZID_USTMv4",
        "pSUBAREAID",
        "aSUBAREAID",
        "PURP7_t",
        "trip_weight",
        "distance_miles",
    ]
]

# filter to HBW
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["PURP7_t"] == "HBW"]

# merge taz
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="pCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "pTAZID"}
)
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="aCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "aTAZID"}
)

# fitler to WF subarea
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["pSUBAREAID"] == 1]
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["aSUBAREAID"] == 1]

# vehicle ownership lookup table
vehown_lookup = (
    hts_veh_23.copy()
    .groupby("hh_id")["vehicle_id"]
    .count()
    .reset_index(name="veh_count")
)
vehown_lookup["vehown"] = vehown_lookup["veh_count"].clip(upper=2)
vehown_lookup = vehown_lookup[["hh_id", "vehown"]]

# income lookup table
income_lookup = hts_hh_23.copy()[["hh_id", "income_detailed"]]
income_lookup["income"] = np.select(
    [
        income_lookup["income_detailed"].isin([1, 2, 3, 4]),
        income_lookup["income_detailed"].isin([5, 6, 7, 8, 9, 10]),
    ],
    ["lo", "hi"],
    default="unknown",
)

income_lookup = income_lookup[["hh_id", "income"]]

# 2. Merge vehicle lookup
hts_trip_23_merge = hts_trip_23_merge.merge(vehown_lookup, how="left", on="hh_id")

# 3. Re-establish 0 vehicles for missing households
# We use Int64 to handle the NaNs during the fill, then keep as Int64 to avoid decimals
hts_trip_23_merge["vehown"] = hts_trip_23_merge["vehown"].fillna(0).astype("Int64")

# 4. Merge income lookup
hts_trip_23_merge = hts_trip_23_merge.merge(income_lookup, how="left", on="hh_id")

# 5. Catch any incomes that failed to merge or were missing
hts_trip_23_merge["income"] = hts_trip_23_merge["income"].fillna("unknown")

# 6. Final Segment Calculation
# Now, a trip only becomes 'unknown' if the income is missing.
# If income is known, it correctly defaults to '0veh_hi', '1veh_lo', etc.
hts_trip_23_merge["segment"] = np.where(
    (hts_trip_23_merge["income"] == "unknown") | (hts_trip_23_merge["income"] == ""),
    "unknown",
    hts_trip_23_merge["vehown"].astype(str)
    + "veh_"
    + hts_trip_23_merge["income"].astype(str),
)


# Added 'segment' back into your final output list so you can see the results!
hts_trip_23_final_w_unknown = hts_trip_23_merge[
    ["segment", "vehown", "income", "pTAZID", "aTAZID", "trip_weight"]
]
hts_trip_23_final = hts_trip_23_final_w_unknown[
    hts_trip_23_final_w_unknown["segment"] != "unknown"
]
hts_trip_23_final


,segment,vehown,income,pTAZID,aTAZID,trip_weight
0,1veh_lo,1,lo,2979.0,2966.0,17.687357
1,2veh_hi,2,hi,2779.0,2983.0,53.679940
2,2veh_hi,2,hi,830.0,866.0,101.865387
4,2veh_hi,2,hi,1909.0,1971.0,292.121640
5,1veh_hi,1,hi,1195.0,1007.0,33.779617
...,...,...,...,...,...,...
14972,1veh_hi,1,hi,512.0,958.0,22.263372
14973,2veh_hi,2,hi,1792.0,1786.0,456.865792
14974,2veh_hi,2,hi,1684.0,1467.0,0.000000
14975,0veh_lo,0,lo,1124.0,1007.0,0.000000


In [ ]:
# double check totals
hts_trip_23_check = hts_trip_23.copy()
hts_trip_23_check = hts_trip_23_check[hts_trip_23_check["PURP7_t"] == "HBW"]
hts_trip_23_check = hts_trip_23_check[hts_trip_23_check["pSUBAREAID"] == 1]
hts_trip_23_check = hts_trip_23_check[hts_trip_23_check["aSUBAREAID"] == 1]

(
    hts_trip_23_final_w_unknown["trip_weight"].sum(),
    hts_trip_23_check["trip_weight"].sum(),
    hts_trip_23_final["trip_weight"].sum(),
)


(np.float64(1711202.513165094),
 np.float64(1711202.513165094),
 np.float64(1511140.4918992128))

## Model Data


### Low, Medium, and High Compensation Employment


In [8]:
# import numpy as np
# import pandas as pd
#
## Note: This assumes df_se now contains the granular columns:
## 'RETL', 'FOOD', 'MANU', 'WSLE', 'GVED', 'OTHR', 'OFFI', 'HLTH'
# df_se = df_se_raw.copy()
#
# if "N" in df_se.columns:
#    df_se_indexed = df_se.set_index(df_se["N"] - 1)
# else:
#    df_se_indexed = df_se.copy()
#
# df_se_full = df_se_indexed.reindex(np.arange(used_zones)).fillna(0)
# if "N" in df_se_full.columns:
#    df_se_full["N"] = np.arange(1, used_zones + 1)
#
## ---------------------------------------------------------
## Aggregate into New Compensation Groups
## ---------------------------------------------------------
# df_se_full["LOWEMP"] = df_se_full["RETL"] + df_se_full["FOOD"]
# df_se_full["MIDEMP"] = (
#    df_se_full["MANU"] + df_se_full["WSLE"] + df_se_full["GVED"] + df_se_full["OTHR"]
# )
# df_se_full["HIGHEMP"] = df_se_full["OFFI"] + df_se_full["HLTH"]
#
## ---------------------------------------------------------
## Employment Ratios
## ---------------------------------------------------------
# denom = (df_se_full["LOWEMP"] + df_se_full["MIDEMP"] + df_se_full["HIGHEMP"]).values
#
# low_pct = np.divide(
#    df_se_full["LOWEMP"].values,
#    denom,
#    out=np.zeros_like(denom, dtype=float),
#    where=denom != 0,
# )
# mid_pct = np.divide(
#    df_se_full["MIDEMP"].values,
#    denom,
#    out=np.zeros_like(denom, dtype=float),
#    where=denom != 0,
# )
# high_pct = np.divide(
#    df_se_full["HIGHEMP"].values,
#    denom,
#    out=np.zeros_like(denom, dtype=float),
#    where=denom != 0,
# )
#
## ---------------------------------------------------------
## Construct Final DataFrame
## ---------------------------------------------------------
# df_se = pd.DataFrame(
#    {
#        "TAZID": df_se_full["N"]
#        if "N" in df_se_full.columns
#        else np.arange(1, used_zones + 1),
#        "LOWEMP": df_se_full["LOWEMP"],
#        "MIDEMP": df_se_full["MIDEMP"],
#        "HIGHEMP": df_se_full["HIGHEMP"],
#        "LOWPCT": low_pct,
#        "MIDPCT": mid_pct,
#        "HIGHPCT": high_pct,
#    }
# )
#
## Optional: Reset index if you want a clean 0-based integer index
# df_se.reset_index(drop=True, inplace=True)
#
## Calculate Total Employment
# df_se["TOTEMP"] = df_se["LOWEMP"] + df_se["MIDEMP"] + df_se["HIGHEMP"]
#
## ---------------------------------------------------------
## Calculate Log Employment
## ---------------------------------------------------------
# df_se["log_job_low"] = np.log(1 + df_se["LOWEMP"])
# df_se["log_job_mid"] = np.log(1 + df_se["MIDEMP"])
# df_se["log_job_high"] = np.log(1 + df_se["HIGHEMP"])
#
# df_se


In [19]:
import numpy as np
import pandas as pd

# Note: This assumes df_se_raw contains the granular columns:
# 'RETL', 'FOOD', 'MANU', 'WSLE', 'GVED', 'OTHR', 'OFFI', 'HLTH'
df_se = df_se_raw.copy()

if "N" in df_se.columns:
    df_se_indexed = df_se.set_index(df_se["N"] - 1)
else:
    df_se_indexed = df_se.copy()

df_se_full = df_se_indexed.reindex(np.arange(used_zones)).fillna(0)
if "N" in df_se_full.columns:
    df_se_full["N"] = np.arange(1, used_zones + 1)

# ---------------------------------------------------------
# Aggregate into 4 New Sector Groups
# ---------------------------------------------------------
# 1. Retail & Hospitality
df_se_full["RETEMP"] = df_se_full["RETL"] + df_se_full["FOOD"]
# 2. Industrial & Logistics
df_se_full["INDEMP"] = df_se_full["MANU"] + df_se_full["WSLE"]
# 3. Office & Institutional
df_se_full["OFFEMP"] = df_se_full["OFFI"] + df_se_full["GVED"] + df_se_full["HLTH"]
# 4. Other
df_se_full["OTHEMP"] = df_se_full["OTHR"]

# ---------------------------------------------------------
# Employment Ratios
# ---------------------------------------------------------
denom = (
    df_se_full["RETEMP"]
    + df_se_full["INDEMP"]
    + df_se_full["OFFEMP"]
    + df_se_full["OTHEMP"]
).values

ret_pct = np.divide(
    df_se_full["RETEMP"].values,
    denom,
    out=np.zeros_like(denom, dtype=float),
    where=denom != 0,
)
ind_pct = np.divide(
    df_se_full["INDEMP"].values,
    denom,
    out=np.zeros_like(denom, dtype=float),
    where=denom != 0,
)
off_pct = np.divide(
    df_se_full["OFFEMP"].values,
    denom,
    out=np.zeros_like(denom, dtype=float),
    where=denom != 0,
)
oth_pct = np.divide(
    df_se_full["OTHEMP"].values,
    denom,
    out=np.zeros_like(denom, dtype=float),
    where=denom != 0,
)

# ---------------------------------------------------------
# Construct Final DataFrame
# ---------------------------------------------------------
df_se = pd.DataFrame(
    {
        "TAZID": df_se_full["N"]
        if "N" in df_se_full.columns
        else np.arange(1, used_zones + 1),
        "RETEMP": df_se_full["RETEMP"],
        "INDEMP": df_se_full["INDEMP"],
        "OFFEMP": df_se_full["OFFEMP"],
        "OTHEMP": df_se_full["OTHEMP"],
        "RETPCT": ret_pct,
        "INDPCT": ind_pct,
        "OFFPCT": off_pct,
        "OTHPCT": oth_pct,
    }
)

# Optional: Reset index if you want a clean 0-based integer index
df_se.reset_index(drop=True, inplace=True)

# Calculate Total Employment
df_se["TOTEMP"] = df_se["RETEMP"] + df_se["INDEMP"] + df_se["OFFEMP"] + df_se["OTHEMP"]

# ---------------------------------------------------------
# Calculate Log Employment
# ---------------------------------------------------------
df_se["log_job_ret"] = np.log(1 + df_se["RETEMP"])
df_se["log_job_ind"] = np.log(1 + df_se["INDEMP"])
df_se["log_job_off"] = np.log(1 + df_se["OFFEMP"])
df_se["log_job_oth"] = np.log(1 + df_se["OTHEMP"])

df_se


,TAZID,RETEMP,INDEMP,OFFEMP,OTHEMP,RETPCT,INDPCT,OFFPCT,OTHPCT,TOTEMP,log_job_ret,log_job_ind,log_job_off,log_job_oth
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
4,5,0.0,0.0,2.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,1.098612,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3624,3625,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
3625,3626,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
3626,3627,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
3627,3628,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0


### Highway Distance


In [20]:
# ---------------------------------------------------------
# 1. Define the new distance impedance function
# ---------------------------------------------------------
def calc_dist_decay(x):
    L = 0.12
    U = 2.0
    x0 = 12.0
    k = 3.0
    # NumPy will seamlessly apply this math to every single item in the array at once
    return L + (U - L) / (1 + (x / x0) ** k)


# ---------------------------------------------------------
# 2. Load Matrix and Build Skim DataFrame
# ---------------------------------------------------------
with omx.open_file(path_skm_hwy, "r") as skm_hwy:
    dist_mtx = np.array(skm_hwy["dist_GP"][:])

# Get matrix dimensions
n_zones = dist_mtx.shape[0]

# Create I and J arrays (Assumes 1-based TAZ IDs)
i_idx = np.arange(1, n_zones + 1)
j_idx = np.arange(1, n_zones + 1)

# Create 2D coordinate grids for I and J
I_grid, J_grid = np.meshgrid(i_idx, j_idx, indexing="ij")

# Flatten the distance matrix once so we aren't calling .ravel() multiple times
dist_flat = dist_mtx.ravel()

# Build the DataFrame using the new formula!
df_skim = pd.DataFrame(
    {
        "I": I_grid.ravel(),
        "J": J_grid.ravel(),
        "dist": dist_flat,
        "dist_decay": calc_dist_decay(dist_flat),  # <-- Replaces ln_dist & short_trip
    }
)

df_skim


,I,J,dist,dist_decay
0,1,1,0.48,1.999880
1,1,2,0.50,1.999864
2,1,3,12.22,1.034390
3,1,4,13.07,0.940222
4,1,5,2.05,1.990674
...,...,...,...,...
13169636,3629,3625,61.94,0.133572
13169637,3629,3626,46.48,0.151805
13169638,3629,3627,23.52,0.340411
13169639,3629,3628,2.11,1.989835


## Integrate Data


In [21]:
# -----------------------------
# LOAD DATA
# -----------------------------
trips = hts_trip_23_final.copy()
zones = df_se.copy()

if "Z" in zones.columns:
    zones = zones.rename(columns={"Z": "TAZID"})

skim = df_skim.copy()

# -----------------------------
# 1. KEEP ONLY ZONES WITH EMPLOYMENT
# -----------------------------
zones = zones[zones["TOTEMP"] > 0].copy()

# -----------------------------
# 2. FILTER TO INTERNAL ZONES
# -----------------------------
zones = zones[zones["TAZID"] < 3563].copy()
zone_ids = zones["TAZID"].values

# -----------------------------
# 3. FILTER TRIPS (optional but recommended)
# Keep only trips whose observed destination is valid
# -----------------------------
trips = trips[trips["aTAZID"].isin(zone_ids)].copy()
print(f"Total valid trips: {len(trips)}")

# -----------------------------
# 4. CROSS JOIN (FULL INTEGRATION)
# -----------------------------
trips["key"] = 1
zones["key"] = 1

choice_df = trips.merge(zones, on="key", suffixes=("", "_dest")).drop("key", axis=1)

# -----------------------------
# 5. MERGE SKIM
# -----------------------------
choice_df = choice_df.merge(
    skim,
    left_on=["pTAZID", "TAZID"],  # origin → destination
    right_on=["I", "J"],
    how="left",
)

# -----------------------------
# 6. CREATE VARIABLES (optional but helpful)
# -----------------------------
choice_df["short_trip"] = (choice_df["dist"] < 6).astype(int)

choice_df["log_job_ret"] = np.log(np.maximum(1, choice_df["RETEMP"]))
choice_df["log_job_ind"] = np.log(np.maximum(1, choice_df["INDEMP"]))
choice_df["log_job_off"] = np.log(np.maximum(1, choice_df["OFFEMP"]))
choice_df["log_job_oth"] = np.log(np.maximum(1, choice_df["OTHEMP"]))

# -----------------------------
# DONE
# -----------------------------
choice_df.to_csv("integrated_data.csv", index=False)


Total valid trips: 13620


In [9]:
# sample size
choice_sample = 50

# -----------------------------
# LOAD DATA
# -----------------------------
trips = hts_trip_23_final.copy()
zones = df_se.copy()

if "Z" in zones.columns:
    zones = zones.rename(columns={"Z": "TAZID"})

skim = df_skim.copy()

# -----------------------------
# 1. KEEP ONLY ZONES WITH EMPLOYMENT
# -----------------------------
zones = zones[zones["TOTEMP"] > 0].copy()

# -----------------------------
# 2. CREATE ZONE LIST FOR SAMPLING
# -----------------------------
zones_ii = zones[zones["TAZID"] < 3563]
zone_ids = zones_ii["TAZID"].values

# 🔥 NEW: FILTER TRIPS BEFORE SAMPLING 🔥
# Drop any observed trips where the chosen destination has 0 jobs or is external.
# This prevents log(0) errors and ensures the chosen alt is in our valid pool.
trips = trips[trips["aTAZID"].isin(zone_ids)].copy()
print(f"Total valid trips for estimation: {len(trips)}")


# -----------------------------
# SAMPLE FUNCTION
# -----------------------------
def sample_alternatives(row, n_sample=choice_sample):
    chosen = row["aTAZID"]

    # 1. Remove chosen from sampling pool
    pool = zone_ids[zone_ids != chosen]
    sampled = np.random.choice(pool, size=n_sample, replace=False)

    # 2. Include chosen alternative
    alts = np.append(sampled, chosen)

    # 3. Create a DataFrame from the ENTIRE row and repeat it 51 times
    # This preserves income, veh, trip_id, etc.
    df = pd.DataFrame([row] * len(alts))

    # 4. Overwrite the 'aTAZID' column with the sampled alternatives
    df["aTAZID"] = alts

    # 5. Mark choice
    df["CHOICE"] = (df["aTAZID"] == chosen).astype(int)

    return df


# -----------------------------
# BUILD CHOICE SET
# -----------------------------
sampled_list = []
for _, row in trips.iterrows():
    sampled_list.append(sample_alternatives(row, choice_sample))

choice_df = pd.concat(sampled_list, ignore_index=True)

# -----------------------------
# MERGE ZONAL DATA (destination side)
# -----------------------------
choice_df = choice_df.merge(zones, left_on="aTAZID", right_on="TAZID", how="left")

# -----------------------------
# MERGE SKIM
# -----------------------------
choice_df = choice_df.merge(
    skim, left_on=["pTAZID", "aTAZID"], right_on=["I", "J"], how="left"
)

# -----------------------------
# CREATE VARIABLES
# -----------------------------
choice_df["short_trip"] = (choice_df["dist"] < 6).astype(int)

# 🔥 NEW: Create the 4-bin log employment variables for Biogeme 🔥
# We use np.maximum(1, ...) just to be absolutely bulletproof against log(0)
choice_df["log_job_ret"] = np.log(np.maximum(1, choice_df["RETEMP"]))
choice_df["log_job_ind"] = np.log(np.maximum(1, choice_df["INDEMP"]))
choice_df["log_job_off"] = np.log(np.maximum(1, choice_df["OFFEMP"]))
choice_df["log_job_oth"] = np.log(np.maximum(1, choice_df["OTHEMP"]))

# unique observation id
choice_df["obs_id"] = np.repeat(np.arange(len(trips)), choice_sample + 1)

# -----------------------------
# SAVE FOR BIOGEME
# -----------------------------
print(choice_df.head())


Total valid trips for estimation: 13620
   vehown income  pTAZID  aTAZID  trip_weight  CHOICE  TAZID  RETEMP  INDEMP  \
0       1     lo  2979.0   380.0    17.687357       0    380     1.0     6.0   
1       1     lo  2979.0  3308.0    17.687357       0   3308     0.0     2.0   
2       1     lo  2979.0   244.0    17.687357       0    244     4.0     0.0   
3       1     lo  2979.0  2120.0    17.687357       0   2120   912.0   450.0   
4       1     lo  2979.0    79.0    17.687357       0     79   177.0     0.0   

   OFFEMP  ...  log_job_ret  log_job_ind  log_job_off  log_job_oth     I  \
0    13.0  ...     0.000000     1.791759     2.564949     2.639057  2979   
1     3.0  ...     0.000000     0.693147     1.098612     0.000000  2979   
2     0.0  ...     1.386294     0.000000     0.000000     0.000000  2979   
3  1422.0  ...     6.815640     6.109248     7.259820     6.887553  2979   
4   158.0  ...     5.176150     0.000000     5.062595     3.044522  2979   

      J    dist  dist_

In [10]:
choice_df.to_csv("choice_data2.csv", index=False)


## Estimate Coefficients & Constants


In [ ]:
## Using popsim environment
# print("Starting Biogeme estimation...")
#
# import pandas as pd
# import numpy as np
# import biogeme.database as db
# import biogeme.biogeme as bio
# import biogeme.models as models
# from biogeme.expressions import Beta, Variable
#
## ==================================================
## 1. LOAD AND PREPARE DATA
## ==================================================
# df = pd.read_csv("choice_data2.csv")
# print("Original data shape:", df.shape)
#
## Map strings to numbers
# income_map = {"lo": 0, "hi": 1}
# if df["income"].dtype == "object":
#    df["income"] = df["income"].map(income_map)
#
## Handle NaNs
# df = df.fillna(0)
#
## ==================================================
## 2. CONVERT LONG TO WIDE FORMAT
## ==================================================
# df["alt_seq"] = df.groupby("obs_id").cumcount() + 1
# chosen_alts = df[df["CHOICE"] == 1].set_index("obs_id")["alt_seq"]
#
## Intrazonal removed
# varying_cols = [
#    "ln_dist",
#    "short_trip",
#    "log_job_low",
#    "log_job_mid",
#    #  "log_job_high",
# ]
#
## Constant columns (including weight)
# constant_cols = ["income", "vehown", "trip_weight"]
#
## Pivot varying columns
# df_wide = df.pivot(index="obs_id", columns="alt_seq", values=varying_cols)
# df_wide.columns = [f"{col[0]}_{col[1]}" for col in df_wide.columns]
#
## Constant columns (including weight)
# df_const = df.drop_duplicates(subset="obs_id").set_index("obs_id")[constant_cols]
# df_wide = df_wide.join(df_const)
#
## Master choice variable
# df_wide["CHOICE_VAR"] = chosen_alts
# df_wide = df_wide.reset_index()
#
## Initialize Biogeme Database
# database = db.Database("dc_model_wide", df_wide)
#
## ==================================================
## 3. SEGMENTATION MASKS
## ==================================================
# INCGRP = Variable("income")
# VEHGRP = Variable("vehown")
#
## Vehicle masks
# V0 = VEHGRP == 0
# V1 = VEHGRP == 1
# V2 = VEHGRP == 2
#
## Income masks
# INC_L = INCGRP == 0
# INC_H = INCGRP == 1
#
## ==================================================
## 4. PARAMETERS (Betas)
## ==================================================
# beta_short = Beta("beta_short", 0, None, None, 0)
#
## Distance parameters (Segmented by Vehicle)
# b_dist_v0 = Beta("b_dist_v0", 0, None, None, 0)
# b_dist_v1 = Beta("b_dist_v1", 0, None, None, 0)
# b_dist_v2 = Beta("b_dist_v2", 0, None, None, 0)
#
## Employment parameters (Segmented by Income)
# b_low_L = Beta("b_low_L", 0, None, None, 0)
# b_mid_L = Beta("b_mid_L", 0, None, None, 0)
## b_high_L = Beta("b_high_L", 0, None, None, 0)
#
# b_low_H = Beta("b_low_H", 0, None, None, 0)
# b_mid_H = Beta("b_mid_H", 0, None, None, 0)
## b_high_H = Beta("b_high_H", 0, None, None, 0)
#
## ==================================================
## 5. UTILITY DICTIONARY (For 51 alternatives)
## ==================================================
# V = {}
# av = {}
#
# for i in range(1, 52):
#    V[i] = (
#        beta_short * Variable(f"short_trip_{i}")
#        + (b_dist_v0 * V0 + b_dist_v1 * V1 + b_dist_v2 * V2) * Variable(f"ln_dist_{i}")
#        + (b_low_L * INC_L + b_low_H * INC_H) * Variable(f"log_job_low_{i}")
#        + (b_mid_L * INC_L + b_mid_H * INC_H) * Variable(f"log_job_mid_{i}")
#        # + (b_high_L * INC_L + b_high_H * INC_H) * Variable(f"log_job_high_{i}")
#    )
#    av[i] = 1
#
## write out file befroe estimation#
## ==================================================
## 5.5 WRITE OUT FILE BEFORE ESTIMATION
## ==================================================
# print("Exporting wide-format dataset before estimation...")
# df_wide.to_csv("pre_estimation_data_wide.csv", index=False)
# print("Data saved to 'pre_estimation_data_wide.csv'.")
#
#
## ==================================================
## 6. ESTIMATION (WITH WEIGHT - VERSION SAFE)
## ==================================================
# CHOICE_VAR = Variable("CHOICE_VAR")
#
## Define weight variable
# WEIGHT = Variable("trip_weight")
#
## Apply weight manually
# logprob = WEIGHT * models.loglogit(V, av, CHOICE_VAR)
#
## Create BIOGEME object (NO weight argument here)
# biogeme = bio.BIOGEME(
#    database, logprob, generate_html=True, generate_yaml=False, generate_netcdf=False
# )
#
# biogeme.modelName = "destination_choice_wide_weighted"
#
## Estimate parameters
# results = biogeme.estimate()
#
## ==================================================
## 7. OUTPUT
## ==================================================
# print("Estimated parameters:")
# print(results.get_estimated_parameters())


Starting Biogeme estimation...
Original data shape: (707931, 24)
Exporting wide-format dataset before estimation...
Data saved to 'pre_estimation_data_wide.csv'.


In [1]:
# Using popsim environment
print("Starting Biogeme estimation...")

import pandas as pd
import numpy as np
import biogeme.database as db
import biogeme.biogeme as bio
import biogeme.models as models
from biogeme.expressions import Beta, Variable

# ==================================================
# 1. LOAD AND PREPARE DATA
# ==================================================
df = pd.read_csv("choice_data2.csv")
print("Original data shape:", df.shape)

# Map strings to numbers
income_map = {"lo": 0, "hi": 1}
if df["income"].dtype == "object":
    df["income"] = df["income"].map(income_map)

# Handle NaNs
df = df.fillna(0)

# ==================================================
# 2. CONVERT LONG TO WIDE FORMAT
# ==================================================
df["alt_seq"] = df.groupby("obs_id").cumcount() + 1
chosen_alts = df[df["CHOICE"] == 1].set_index("obs_id")["alt_seq"]

# Updated to the new 4-bin employment groups + dist_decay
varying_cols = [
    "dist_decay",
    "log_job_ret",
    "log_job_ind",
    "log_job_off",
    # "log_job_oth",  # Left out to serve as the reference group
]

# Constant columns (including weight)
constant_cols = ["income", "vehown", "trip_weight"]

# Pivot varying columns
df_wide = df.pivot(index="obs_id", columns="alt_seq", values=varying_cols)
df_wide.columns = [f"{col[0]}_{col[1]}" for col in df_wide.columns]

# Constant columns (including weight)
df_const = df.drop_duplicates(subset="obs_id").set_index("obs_id")[constant_cols]
df_wide = df_wide.join(df_const)

# Master choice variable
df_wide["CHOICE_VAR"] = chosen_alts
df_wide = df_wide.reset_index()

# Initialize Biogeme Database
database = db.Database("dc_model_wide", df_wide)

# ==================================================
# 3. SEGMENTATION MASKS
# ==================================================
INCGRP = Variable("income")
VEHGRP = Variable("vehown")

# Vehicle masks
V0 = VEHGRP == 0
V1 = VEHGRP == 1
V2 = VEHGRP == 2

# Income masks
INC_L = INCGRP == 0
INC_H = INCGRP == 1

# ==================================================
# 4. PARAMETERS (Betas)
# ==================================================
# Distance parameters (Segmented by Vehicle)
# b_dist_v0 is fixed to 0 (last argument is 1)
b_dist_v0 = Beta("b_dist_v0", 0, None, None, 1)
b_dist_v1 = Beta("b_dist_v1", 0, None, None, 0)
b_dist_v2 = Beta("b_dist_v2", 0, None, None, 0)

# Employment parameters (Segmented by Income)
# b_ret_L is fixed to 0 (last argument is 1)
b_ret_L = Beta("b_ret_L", 0, None, None, 1)
b_ind_L = Beta("b_ind_L", 0, None, None, 0)
b_off_L = Beta("b_off_L", 0, None, None, 0)
# b_oth_L = Beta("b_oth_L", 0, None, None, 0)

b_ret_H = Beta("b_ret_H", 0, None, None, 0)
b_ind_H = Beta("b_ind_H", 0, None, None, 0)
b_off_H = Beta("b_off_H", 0, None, None, 0)
# b_oth_H = Beta("b_oth_H", 0, None, None, 0)

# ==================================================
# 5. UTILITY DICTIONARY (For 51 alternatives)
# ==================================================
V = {}
av = {}

for i in range(1, 52):
    V[i] = (
        (b_dist_v0 * V0 + b_dist_v1 * V1 + b_dist_v2 * V2) * Variable(f"dist_decay_{i}")
        + (b_ret_L * INC_L + b_ret_H * INC_H) * Variable(f"log_job_ret_{i}")
        + (b_ind_L * INC_L + b_ind_H * INC_H) * Variable(f"log_job_ind_{i}")
        + (b_off_L * INC_L + b_off_H * INC_H) * Variable(f"log_job_off_{i}")
        # + (b_oth_L * INC_L + b_oth_H * INC_H) * Variable(f"log_job_oth_{i}")
    )
    av[i] = 1

## ==================================================
## 5.5 WRITE OUT FILE BEFORE ESTIMATION
## ==================================================
# print("Exporting wide-format dataset before estimation...")
# df_wide.to_csv("pre_estimation_data_wide.csv", index=False)
# print("Data saved to 'pre_estimation_data_wide.csv'.")

# ==================================================
# 6. ESTIMATION (WITH WEIGHT - VERSION SAFE)
# ==================================================
CHOICE_VAR = Variable("CHOICE_VAR")

# Define weight variable
WEIGHT = Variable("trip_weight")

# Apply weight manually
logprob = WEIGHT * models.loglogit(V, av, CHOICE_VAR)

# Create BIOGEME object (NO weight argument here)
biogeme = bio.BIOGEME(
    database, logprob, generate_html=True, generate_yaml=False, generate_netcdf=False
)

# 1. Use the new model_name syntax to silence the warning
# 2. Change the name so it ignores your old cached results!
biogeme.model_name = "destination_choice_fixed_params"

# Estimate parameters
results = biogeme.estimate()

# ==================================================
# 7. OUTPUT
# ==================================================
print("Estimated parameters:")
print(results.get_estimated_parameters())


Starting Biogeme estimation...


WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
c:\Users\cday\Anaconda3\envs\base0426\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Original data shape: (694620, 26)
Estimated parameters:
        Name     Value  Robust std err.  Robust t-stat.  Robust p-value
0  b_dist_v1  1.996296         0.110305       18.098030    0.000000e+00
1  b_dist_v2  1.968287         0.032336       60.870751    0.000000e+00
2    b_ret_H -0.052676         0.013994       -3.764061    1.671763e-04
3    b_ind_L  0.158804         0.023526        6.750235    1.476064e-11
4    b_ind_H  0.184105         0.013762       13.378114    0.000000e+00
5    b_off_L  0.414622         0.034824       11.906089    0.000000e+00
6    b_off_H  0.547637         0.019027       28.782045    0.000000e+00


C:\Users\cday\AppData\Local\Temp\ipykernel_37952\3504522635.py:144: DeprecationWarning: get_estimated_parameters is deprecated. Use get_pandas_estimated_parameters(estimation_results=my_results) instead
  print(results.get_estimated_parameters())
